# MCDD experiments

This notebook runs the experimental workflow used in the MCDD study on the HDF5 streams stored under `data/datasets/`.

Included methods: MCDD, Traditional Single Hypothesis (TSH), River KSWIN, and LORD under local dependence.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src' / 'mcdd').is_dir():
            return candidate
    raise FileNotFoundError('Repository root not found.')


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / 'src'
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

from mcdd.experiments import (
    PAPER_CONFIGURATIONS,
    article_table_for_drift,
    article_table_to_latex,
    average_metrics_by_drift,
    average_metrics_overall,
    configuration_table,
    evaluate_single_stream,
    expected_dataset_paths,
    format_article_table,
    overall_article_table,
    read_stream,
    run_experiment_suite,
    summarize_results,
)

DATA_DIRECTORY = REPOSITORY_ROOT / 'data' / 'datasets'
RESULTS_DIRECTORY = REPOSITORY_ROOT / 'results'
PER_RUN_RESULTS = RESULTS_DIRECTORY / 'per_run_results.csv'
SUMMARY_RESULTS = RESULTS_DIRECTORY / 'summary_results.csv'

print(f'Repository root: {REPOSITORY_ROOT}')
print(f'Dataset directory: {DATA_DIRECTORY}')


## Scoring convention

Each stream contains one known concept drift. The first alarm is classified as follows:

- `alarm_index < drift_start`: false alarm (FP);
- `drift_start <= alarm_index <= valid_detection_end`: valid detection (TP);
- `alarm_index > valid_detection_end`: late detection (FN);
- no alarm: missed detection (FN).

Abrupt drift uses a 2,000-sample detection margin. Gradual and incremental drift use the generated transition interval. Late detections remain identifiable through `outcome='late_detection'` and `late_delay`.

The summary metrics are `FDR = FP / (TP + FP)`, `MDR = FN / (TP + FN)`, and `IR = TP / (TP + FP)`. Undefined ratios and Mean Delay without valid detections are stored as `NaN`.


In [ ]:
display(configuration_table(PAPER_CONFIGURATIONS))

for dataset_path in expected_dataset_paths(DATA_DIRECTORY):
    print(dataset_path.name)


## Quick validation

Run all detector configurations on the first stream of `abrupt_normal.h5`.


In [ ]:
quick_archive = DATA_DIRECTORY / 'abrupt_normal.h5'
values, metadata = read_stream(quick_archive, row_index=0)
quick_results = [
    evaluate_single_stream(values, **metadata, configuration=configuration)
    for configuration in PAPER_CONFIGURATIONS
]
display(pd.DataFrame(quick_results)[[
    'configuration', 'alarm_index', 'outcome', 'drift_start',
    'valid_detection_end', 'delay', 'late_delay', 'TP', 'FP', 'FN'
]])


## Full experiment execution

The complete benchmark evaluates 9 HDF5 archives × 1,000 streams × 10 detector configurations = 90,000 detector-stream runs.


In [ ]:
RUN_FULL_EXPERIMENTS = False
OVERWRITE_RESULTS = False
MAX_STREAMS_PER_ARCHIVE = None

if RUN_FULL_EXPERIMENTS:
    run_experiment_suite(
        data_directory=DATA_DIRECTORY,
        output_file=PER_RUN_RESULTS,
        configurations=PAPER_CONFIGURATIONS,
        max_streams_per_archive=MAX_STREAMS_PER_ARCHIVE,
        overwrite=OVERWRITE_RESULTS,
        progress_every=25,
    )
    summary = summarize_results(
        per_run_file=PER_RUN_RESULTS,
        output_file=SUMMARY_RESULTS,
    )
    display(summary)
else:
    print('Full execution is disabled.')


## Inspect results and article-style tables


In [ ]:
if PER_RUN_RESULTS.is_file():
    per_run_results = pd.read_csv(PER_RUN_RESULTS)
    print(f'Per-run rows: {len(per_run_results):,}')
    display(per_run_results.head())

    summary = summarize_results(
        per_run_file=PER_RUN_RESULTS,
        output_file=SUMMARY_RESULTS,
    )
    display(summary)

    by_drift = average_metrics_by_drift(
        SUMMARY_RESULTS,
        article_na=True,
        require_all_distributions=False,
    )
    for drift_type in ('abrupt', 'gradual', 'incremental'):
        table = article_table_for_drift(by_drift, drift_type)
        if not table.empty:
            print(drift_type.title())
            display(format_article_table(table))

    overall = average_metrics_overall(by_drift, require_all_drifts=False)
    display(format_article_table(overall_article_table(overall)))
else:
    print(f'No per-run result file found at {PER_RUN_RESULTS}.')
